# Study 867 — Currency Crash Risk — the teardown

The basket crash skew (Newey-West *t* on the standardised-cubed residuals), the skew-carry cross-section (slope + Spearman), the label-shuffle placebo, the crash-conditional split, the two-era cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2003-12-12', 'end': '2026-06-26', 'n_ccy': 8, 'n_weeks': 1177, 'basket_skew': -1.39, 'skew_t': -1.51, 'premium_bps': 6.33, 'premium_ann': 3.29, 'premium_t': 1.73, 'sharpe': 0.34, 'worst_week': -13.3, 'max_dd': -36.0, 'calm_ann': 12.6, 'off_ann': -173.6, 'slope': -0.312, 't_slope': -2.41, 'r2': 0.49, 'spearman': -0.833, 'spearman_p': 0.008, 'hi_leg_skew': -1.08, 'lo_leg_skew': 0.38, 'leg_diff': -1.46, 'placebo_obs': -1.39, 'placebo_mean': -0.001, 'placebo_sd': 0.815, 'placebo_p': 0.0335, 'era_early_skew': -1.41, 'era_early_skew_t': -1.21, 'era_early_prem': 3.7, 'era_early_prem_t': 1.19, 'era_early_slope_t': -3.39, 'era_early_spearman': -0.786, 'era_early_n': 577, 'era_late_skew': -1.11, 'era_late_skew_t': -1.92, 'era_late_prem': 2.9, 'era_late_prem_t': 1.3, 'era_late_slope_t': -1.58, 'era_late_spearman': -0.833, 'era_late_n': 600, 'timer_0_net': 1.21, 'timer_0_sh': 0.13, 'timer_50_net': 0.71, 'timer_50_sh': 0.07, 'timer_50_t': 0.37, 'timer_100_net': 0.21, 'null_skew_t_mean': 0.02, 'null_skew_t_sd': 0.73, 'null_fire': 0, 'null_slope_mean': 0.0019, 'planted_skew': -3.5, 'planted_slope': -0.696, 'planted_slope_t': -9.98, 'planted_spearman': -0.976}

## The headline — the carry basket's crash skew + premium

Dollar-neutral long top-3 / short bottom-3 carry basket, weekly.

In [2]:
print(f"realized skew : {R['basket_skew']:+.2f}   NW(6) skew t = {R['skew_t']:+.2f}")
print(f"premium       : {R['premium_bps']:+.2f} bps/wk ({R['premium_ann']:+.2f}%/yr)  "
      f"NW t = {R['premium_t']:+.2f}  Sharpe {R['sharpe']:.2f}")
print(f"crash shape   : worst week {R['worst_week']:+.1f}%  max DD {R['max_dd']:+.1f}%")
print(f"crash split   : calm {R['calm_ann']:+.1f}%/yr vs worst-5% weeks {R['off_ann']:+.1f}%/yr")

realized skew : -1.39   NW(6) skew t = -1.51
premium       : +6.33 bps/wk (+3.29%/yr)  NW t = +1.73  Sharpe 0.34
crash shape   : worst week -13.3%  max DD -36.0%
crash split   : calm +12.6%/yr vs worst-5% weeks -173.6%/yr


## The skew-carry cross-section — higher carry, more negative skew?

In [3]:
print(f"slope(skew on carry) = {R['slope']:+.3f}  t = {R['t_slope']:+.2f}  R2 = {R['r2']:.2f}")
print(f"Spearman rank corr   = {R['spearman']:+.3f}  (permutation p = {R['spearman_p']:.3f})")
print(f"high-carry leg skew {R['hi_leg_skew']:+.2f} vs low-carry leg skew {R['lo_leg_skew']:+.2f} "
      f"(diff {R['leg_diff']:+.2f})")

slope(skew on carry) = -0.312  t = -2.41  R2 = 0.49
Spearman rank corr   = -0.833  (permutation p = 0.008)
high-carry leg skew -1.08 vs low-carry leg skew +0.38 (diff -1.46)


## Placebo — shuffle which currency owns which carry (2,000 relabelings)

In [4]:
print(f"observed basket skew {R['placebo_obs']:+.2f} vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> left-tail p = {R['placebo_p']:.4f}")

observed basket skew -1.39 vs placebo mean -0.001 (sd 0.815) -> left-tail p = 0.0335


## Robustness — two eras (split 2015-01-01)

In [5]:
print(f"2004-2014 (n={R['era_early_n']}): basket skew {R['era_early_skew']:+.2f} (t {R['era_early_skew_t']:+.2f}) "
      f"| premium {R['era_early_prem']:+.2f}%/yr (t {R['era_early_prem_t']:+.2f}) "
      f"| skew-carry slope t {R['era_early_slope_t']:+.2f} Spearman {R['era_early_spearman']:+.3f}")
print(f"2015-2026 (n={R['era_late_n']}): basket skew {R['era_late_skew']:+.2f} (t {R['era_late_skew_t']:+.2f}) "
      f"| premium {R['era_late_prem']:+.2f}%/yr (t {R['era_late_prem_t']:+.2f}) "
      f"| skew-carry slope t {R['era_late_slope_t']:+.2f} Spearman {R['era_late_spearman']:+.3f}")

2004-2014 (n=577): basket skew -1.41 (t -1.21) | premium +3.70%/yr (t +1.19) | skew-carry slope t -3.39 Spearman -0.786
2015-2026 (n=600): basket skew -1.11 (t -1.92) | premium +2.90%/yr (t +1.30) | skew-carry slope t -1.58 Spearman -0.833


## The timer — can you get paid for the crash risk?

Costed carry book: 2 bps/side rebalance + borrow on the short leg.

In [6]:
print(f"borrow   0 bps/yr: net {R['timer_0_net']:+.2f}%/yr  Sharpe {R['timer_0_sh']:.2f}")
print(f"borrow  50 bps/yr: net {R['timer_50_net']:+.2f}%/yr  Sharpe {R['timer_50_sh']:.2f}  (t {R['timer_50_t']:+.2f})")
print(f"borrow 100 bps/yr: net {R['timer_100_net']:+.2f}%/yr")

borrow   0 bps/yr: net +1.21%/yr  Sharpe 0.13
borrow  50 bps/yr: net +0.71%/yr  Sharpe 0.07  (t +0.37)
borrow 100 bps/yr: net +0.21%/yr


## Synthetic positive control — the machinery is unbiased

Live: the powered skew-carry detector must NOT fire on the null and must recover a planted carry-crash relation. (The basket *skew t* is deliberately low-powered against rare crashes — even a planted skew of −3.5 returns skew t ≈ −1.)

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from fx_crash import data, strategy as st
null_slope = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=867+s, n_weeks=1000))['slope'] for s in range(8)])
print(f"null (edge=0), 8 seeds: skew-carry slope mean {null_slope.mean():+.4f} (sd {null_slope.std(ddof=1):.4f})")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.02, seed=867, n_weeks=1000))
print(f"planted (edge=0.02): basket skew {planted['basket_skew']:+.2f}, skew-carry slope {planted['slope']:+.3f} (Spearman {planted['spearman']:+.3f})")

null (edge=0), 8 seeds: skew-carry slope mean -0.0049 (sd 0.0083)
planted (edge=0.02): basket skew -1.33, skew-carry slope -0.277 (Spearman -0.976)


## Verdict

- **Signal — Weak.** The BNP crash-skew signature is genuinely present and correctly signed: basket skew **-1.39** (label-shuffle *p* = 0.034), a strongly monotone skew-carry cross-section (Spearman **-0.83**, permutation *p* = 0.008), stable in sign across both eras, textbook crash accounting (worst week -13.3%, max DD -36.0%). But the strict green bar — a robust Newey-West |t| ≥ 2 holding across sub-eras — is not cleared: the basket-skew NW *t* is only **-1.51** (-1.21 / -1.92 by era) and the slope *t* falls to -1.58 late. The 20-seed synthetic control fires on 0/20 nulls and recovers a planted relation cleanly, so the borderline real result is honest, not machinery.
- **Tradability — Mirage.** The premium is weak (**+3.29%/yr**, *t* = +1.73) and collapses to **+0.71%/yr** (Sharpe 0.07) at 50 bps/yr borrow — pennies in front of a −36% steamroller.